# 01 · The broker and its monitoring programme

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohammedelsayed741996-blip/berlin-data-portfolio/blob/main/fintech-brokerage/aml-monitoring-oversight/notebooks/01_programme_and_data.ipynb)

A fictional mobile broker in Germany with 15,000 clients, a year of deposits, trades and withdrawals (October 2025 to September 2026), six transaction monitoring rules, name screening against a sanctions and PEP list, and a team of eight first-line analysts.

The oversight team (the second line) does not run the monitoring. It checks that the monitoring works and reports to the MLRO, senior management and the board. This notebook shows what there is to check. Notebooks 02 and 03 do the checking.

Everything is synthetic. Names are random syllables, accounts are random hashes, analysts are codes, and HR1 to HR4 stand in for high-risk jurisdictions.

In [1]:
import os, sys, subprocess, warnings
from pathlib import Path

REPO = "https://github.com/mohammedelsayed741996-blip/berlin-data-portfolio.git"
PROJECT = "fintech-brokerage/aml-monitoring-oversight"

if "google.colab" in sys.modules:
    if not Path("/content/berlin-data-portfolio").exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO, "/content/berlin-data-portfolio"], check=True)
    os.chdir(f"/content/berlin-data-portfolio/{PROJECT}")
else:
    here = Path.cwd()
    while not (here / "src" / "aml_oversight").exists() and here != here.parent:
        here = here.parent
    os.chdir(here)

sys.path.insert(0, str(Path.cwd() / "src"))
warnings.filterwarnings("ignore")
import pandas as pd
pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)
from IPython.display import Image, display
print("Project folder:", Path.cwd().name)

Project folder: aml-monitoring-oversight


In [2]:
from aml_oversight import config as C, pipeline, oversight as O, reporting, db

tables = pipeline.load_tables()      # reads data/, or generates it if missing
run = pipeline.run(tables)
print(f"{len(tables['clients']):,} clients, {len(tables['transactions']):,} transactions, "
      f"{len(tables['alerts']):,} alerts, {len(tables['screening_hits']):,} screening hits")

15,000 clients, 327,703 transactions, 1,919 alerts, 1,228 screening hits


## Clients and activity

In [3]:
cl = tables["clients"]
display(cl[["client_id", "residence", "occupation", "expected_monthly_deposit_eur", "risk_rating", "pep"]].head())
display(cl["risk_rating"].value_counts().to_frame().T)
tx = tables["transactions"]
display(tx.groupby("type")["amount_eur"].agg(["count", "sum"]).round(0))

,client_id,residence,occupation,expected_monthly_deposit_eur,risk_rating,pep
0,C000001,DE,employee,298.0,low,False
1,C000002,DE,employee,814.0,low,False
2,C000003,DE,employee,680.0,low,False
3,C000004,DE,employee,361.0,low,False
4,C000005,DE,employee,320.0,low,False


risk_rating,low,medium,high
count,12030,2419,551


,count,sum
type,,
BUY,122967,57326330.0
DEPOSIT,114416,85832435.0
SELL,81487,29302071.0
WITHDRAWAL,8833,7996751.0


## Planted behaviour

Six typologies sit inside the normal activity: structuring just under EUR 10,000, pass-through (money in and straight out), funding from many third parties, mule rings, money from high-risk jurisdictions, and activity far above the client's declared income. Some legitimate clients look unusual too: an inheritance, a monthly saving of EUR 9,000, a parent paying in.

The mule rings are built to slip past transaction rules: small card deposits from the client's own card, a quick buy and sell so the money looks invested, then a payout to an account shared by the whole ring.

In [4]:
display(tables["typology_truth"]["typology"].value_counts().to_frame().T)
mule = tables["typology_truth"].query("typology == 'T4 mule ring'")["client_id"].iloc[0]
tx[tx["client_id"] == mule].head(8)

typology,T4 mule ring,T2 pass-through,T3 third-party funding,T1 structuring,T6 profile mismatch,T5 high-risk jurisdiction
count,36,30,30,25,25,20


,tx_id,client_id,date,type,amount_eur,channel,psp,counterparty,counterparty_country,own_account
27088,T00027089,C002167,2025-11-04,DEPOSIT,186.14,CARD,PSP-B,CARD8c5af23b79c7,DE,True
38638,T00038639,C002167,2025-11-17,BUY,327.62,SEPA,,,DE,False
47503,T00047504,C002167,2025-11-27,BUY,86.16,SEPA,,,DE,False
51991,T00051992,C002167,2025-12-04,DEPOSIT,862.66,CARD,PSP-A,CARD8c5af23b79c7,DE,True
52882,T00052883,C002167,2025-12-05,BUY,845.41,SEPA,,,DE,False
54743,T00054744,C002167,2025-12-07,SELL,836.78,SEPA,,,DE,False
55677,T00055678,C002167,2025-12-08,WITHDRAWAL,828.15,SEPA,,IBAN13c0e8c116e4,DE,True
59360,T00059361,C002167,2025-12-12,DEPOSIT,1368.33,CARD,PSP-A,CARD8c5af23b79c7,DE,True


## The rules and their alerts

In [5]:
a = tables["alerts"]
display(pd.DataFrame({"rule": list(C.RULES), "description": list(C.RULES.values())}))
display(a.groupby("rule").size().rename("alerts").to_frame().T)
a[["alert_id", "client_id", "rule", "alert_date", "analyst", "l1_decision", "l1_rationale", "l2_decision"]].head(8)

,rule,description
0,R01,"Single deposit of EUR 15,000 or more"
1,R02,"Several deposits between EUR 8,500 and 9,999 within 30 days"
2,R03,"Rapid in and out: 80% of deposits withdrawn within 7 days, little trading"
3,R04,Funding from 2 or more third-party accounts within 30 days
4,R05,Deposit from a high-risk jurisdiction
5,R06,"Monthly deposits above 5x the expected amount and above EUR 20,000"


rule,R01,R02,R03,R04,R05,R06
alerts,433,139,80,511,96,660


,alert_id,client_id,rule,alert_date,analyst,l1_decision,l1_rationale,l2_decision
0,AL000001,C006960,R05,2025-10-01,A06,closed_no_concern,Deposits match recurring savings pattern seen over 12 months.,
1,AL000002,C003813,R01,2025-10-02,A06,closed_no_concern,One-off transfer explained by client (property sale); documents on file.,
2,AL000003,C014017,R04,2025-10-02,A04,closed_no_concern,Deposits match recurring savings pattern seen over 12 months.,
3,AL000004,C013586,R04,2025-10-03,A02,escalated,Funds withdrawn shortly after deposit with no investment purpose; escalated.,SAR filed
4,AL000005,C002491,R05,2025-10-04,A04,escalated,Multiple unrelated third-party payers; escalated for source of funds review.,SAR filed
5,AL000006,C014591,R04,2025-10-05,A07,escalated,Funds withdrawn shortly after deposit with no investment purpose; escalated.,SAR filed
6,AL000007,C006855,R04,2025-10-06,A01,closed_no_concern,Third-party deposit from spouse's account; relationship verified.,
7,AL000008,C013298,R04,2025-10-06,A05,closed_no_concern,One-off transfer explained by client (property sale); documents on file.,


## What the case system records

Each alert has a level 1 decision (closed with no concern, or escalated), a date and a written rationale. Escalated alerts get a level 2 decision and, where suspicion is confirmed, a SAR to the FIU. These are the fields the oversight KPIs are built from. The ground-truth column `true_suspicious` exists only because the data is synthetic; the oversight code uses it solely for sampled QA re-reviews.

In [6]:
display(a["l1_decision"].value_counts().to_frame().T)
display(a["l2_decision"].replace("", "not escalated").value_counts().to_frame().T)

l1_decision,closed_no_concern,escalated,open
count,1185,692,42


l2_decision,not escalated,SAR filed,no SAR,in investigation
count,1227,576,114,2


## Screening

In [7]:
h = tables["screening_hits"]
display(h.groupby(["list", "decision"]).size().unstack(fill_value=0))
h.sort_values("match_score", ascending=False)[["client_id", "entry_id", "list", "match_score", "birth_year_match", "decision"]].head(6)

decision,false positive,true match
list,,
PEP,527,57
sanctions,642,2


,client_id,entry_id,list,match_score,birth_year_match,decision
1161,C008483,W0126,sanctions,1.0,False,false positive
1174,C012220,W0182,sanctions,1.0,False,false positive
56,C005386,W0425,PEP,1.0,False,false positive
0,C000035,W0401,PEP,1.0,True,true match
12,C001264,W0407,PEP,1.0,True,true match
71,C006645,W0335,PEP,1.0,False,false positive
